1. Importaciones iniciales

In [1]:
import json
from datetime import date, datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

StatementMeta(, c50529a7-5f01-44c2-b0e4-85a9fe4d0823, 3, Finished, Available, Finished, False)

2. Comprobar el Lakehouse y los archivos de entrada (Bronze)

In [2]:
# 1. Comprobar contexto y archivos Bronze. Cada ejecución del pipeline deja
# un archivo distinto. Volver a leerlos todos hace la demostración reproducible;
# MERGE impide que vuelvan a entrar las fechas ya presentes en Silver.
contexto = notebookutils.runtime.context
assert contexto.get("defaultLakehouseName") == "LH_PreciosCombustibles", (
    "Vincula LH_PreciosCombustibles como Lakehouse predeterminado."
)

carpeta_bronze = Path("/lakehouse/default/Files/bronze/eia")
archivos = sorted(carpeta_bronze.glob("*.json"))
if not archivos:
    raise FileNotFoundError("No hay JSON de EIA en Files/bronze/eia. Ejecuta antes la copia del pipeline.")
print(f"Archivos Bronze encontrados: {len(archivos)}")

StatementMeta(, c50529a7-5f01-44c2-b0e4-85a9fe4d0823, 4, Finished, Available, Finished, False)

Archivos Bronze encontrados: 1


3. Procesamiento y lectura de los archivos JSON

In [3]:
def leer_archivo(archivo):
    with open(archivo, "r", encoding="utf-8") as f:
        datos = json.load(f)
    if isinstance(datos, dict) and "response" in datos:
        return datos["response"].get("data", [])
    elif isinstance(datos, list):
        return datos
    return datos.get("data", [])

series = {
    "EMM_EPMR_PTE_NUS_DPG": "Gasolina regular",
    "EMD_EPD2D_PTE_NUS_DPG": "Diésel",
}

filas = []
invalidas = 0

for archivo in archivos:
    capturado_en = datetime.fromtimestamp(archivo.stat().st_mtime, timezone.utc)
    for registro in leer_archivo(archivo):
        serie = registro.get("series")
        try:
            fecha = date.fromisoformat(str(registro["period"])[:10])
            precio = float(registro["value"])
            if serie not in series or precio <= 0:
                raise ValueError("Serie o precio fuera de alcance")
        except (KeyError, ValueError, TypeError):
            invalidas += 1
            continue
        filas.append((
            serie, 
            series[serie], 
            "Estados Unidos", 
            fecha, 
            precio, 
            "USD/galón", 
            capturado_en, 
            archivo.name, 
            fecha.year, 
            fecha.month
        ))

if not filas:
    raise ValueError("Los archivos Bronze no contienen observaciones válidas de las dos series EIA.")

StatementMeta(, c50529a7-5f01-44c2-b0e4-85a9fe4d0823, 5, Finished, Available, Finished, False)

4. Definición del Esquema y Deduplicación por Serie y Fecha

In [4]:
esquema = T.StructType([
    T.StructField("serie_eia", T.StringType(), False),
    T.StructField("combustible", T.StringType(), False),
    T.StructField("region", T.StringType(), False),
    T.StructField("fecha", T.DateType(), False),
    T.StructField("precio_usd_galon", T.DoubleType(), False),
    T.StructField("unidad", T.StringType(), False),
    T.StructField("capturado_en_utc", T.TimestampType(), False),
    T.StructField("archivo_origen", T.StringType(), False),
    T.StructField("anio", T.IntegerType(), False),
    T.StructField("mes", T.IntegerType(), False)
])

entrada = spark.createDataFrame(filas, esquema)

ventana = Window.partitionBy("serie_eia", "fecha").orderBy(
    F.col("capturado_en_utc").desc(), 
    F.col("archivo_origen").asc()
)

unicas = entrada.withColumn("orden", F.row_number().over(ventana)) \
    .where(F.col("orden") == 1).drop("orden")

total_entrada = entrada.count()
total_unicas = unicas.count()

StatementMeta(, c50529a7-5f01-44c2-b0e4-85a9fe4d0823, 6, Finished, Available, Finished, False)

5. Insertar solo claves nuevas en la Tabla Silver (Delta Lake Merge)

In [5]:
# 4. Crear Silver Delta una sola vez, particionada por año y mes, y agregar
# exclusivamente las claves todavía ausentes. Repetir la carga es idempotente.

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

tabla = "silver.precios_combustibles_historico"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {tabla} (
    serie_eia STRING, 
    combustible STRING, 
    region STRING, 
    fecha DATE,
    precio_usd_galon DOUBLE, 
    unidad STRING, 
    capturado_en_utc TIMESTAMP,
    archivo_origen STRING, 
    anio INT, 
    mes INT
) USING DELTA PARTITIONED BY (anio, mes)
""")

antes = spark.table(tabla).count()

DeltaTable.forName(spark, tabla).alias("destino").merge(
    unicas.alias("origen"),
    "destino.serie_eia = origen.serie_eia AND destino.fecha = origen.fecha"
).whenNotMatchedInsertAll().execute()

despues = spark.table(tabla).count()
print(f"Filas antes: {antes} | Filas después: {despues} | Nuevas agregadas: {despues - antes}")

StatementMeta(, c50529a7-5f01-44c2-b0e4-85a9fe4d0823, 7, Finished, Available, Finished, False)

Filas antes: 0 | Filas después: 3589 | Nuevas agregadas: 3589
